<a href="https://colab.research.google.com/github/Kamaxi74/Stocks-Predictor-Interactive-Dashboard/blob/src/Kamaxi_FINAL_Project_TESLA_GOOGLE_Stock_Price_Prediction_System_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Tesla & Google Stock Price Prediction using LSTM model***

In [ ]:


# Importing required Libraries

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_squared_error,r2_score
from keras.callbacks import EarlyStopping

#Install Streamlit and pyngrok for creating public URL
!pip install -q streamlit
!pip install pyngrok



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 82.2 MB/s eta 0:00:00


***Processing TESLA (TSLA)***

***1. Fetching Data for TSLA***

In [ ]:
print("---1. Fetching Data for TSLA---")
# Download TSLA stock data
tsla_stock_data = yf.download('TSLA', start='2010-01-01', end='2025-01-01')

# Select only the 'Close' column and save it to a CSV file with the date index
tsla_data_series = tsla_stock_data['Close']
tsla_data_series.to_csv("tsla_data.csv", header=['Close'])

# Load the data from the simplified CSV file, specifying header and index column
tsla_data = pd.read_csv("tsla_data.csv", header=0, index_col=0)

# Rename the index to 'Date' for clarity
tsla_data.index.name = 'Date'

tsla_data.head()

---1. Fetching Data for TSLA---


/tmp/ipython-input-2495902612.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  tsla_stock_data = yf.download('TSLA', start='2010-01-01', end='2025-01-01')
[*********************100%***********************]  1 of 1 completed


,Close
Date,
2010-06-29,1.592667
2010-06-30,1.588667
2010-07-01,1.464000
2010-07-02,1.280000
2010-07-06,1.074000


***2. Data Preprocessing -EDA***

In [ ]:
print("\n--- 2. Data Preprocessing & EDA for TSLA ---")
print("\nTSLA Data Summary:")
print(tsla_data.describe())
print("\nTSLA Data Info:")
print(tsla_data.info())

#To Find Null Values
tsla_data.isna().sum()



--- 2. Data Preprocessing & EDA for TSLA ---

TSLA Data Summary:
             Close
count  3652.000000
mean     81.524477
std     107.619429
min       1.053333
25%      12.097667
50%      17.924666
75%     177.909996
max     479.859985

TSLA Data Info:
<class 'pandas.core.frame.DataFrame'>
Index: 3652 entries, 2010-06-29 to 2024-12-31
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   3652 non-null   float64
dtypes: float64(1)
memory usage: 57.1+ KB
None


,0
Close,0


In [ ]:
#Remove Duplicates
tsla_data=tsla_data.drop_duplicates()
tsla_data.duplicated().sum()
tsla_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3446 entries, 2010-06-29 to 2024-12-31
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   3446 non-null   float64
dtypes: float64(1)
memory usage: 53.8+ KB


***2. Feature Scaling- Data Normalization***

In [ ]:
print("\n--- 3. Feature Scaling -Data Normalization for TSLA ---")
# Select only the 'Close' column for scaling
tsla_data_scaled = tsla_data['Close'].values.reshape(-1, 1)
tsla_scaler = MinMaxScaler(feature_range=(0, 1))
tsla_data_scaled = tsla_scaler.fit_transform(tsla_data_scaled)

tsla_data_scaled


--- 3. Feature Scaling -Data Normalization for TSLA ---


array([[1.12641280e-03],
       [1.11805880e-03],
       [8.57688465e-04],
       ...,
       [8.99333099e-01],
       [8.69571608e-01],
       [8.41230299e-01]])

In [ ]:
#Display data in proper Dataframe
Newtsla_data_scaled = pd.DataFrame(tsla_data_scaled, columns = ['Close'] , index = tsla_data.index)
Newtsla_data_scaled.head()

,Close
Date,
2010-06-29,0.001126
2010-06-30,0.001118
2010-07-01,0.000858
2010-07-02,0.000473
2010-07-06,0.000043


***3. Creating the Window Sequences***


In [ ]:
look_back = 60

def create_sequence(data, look_back):
    X = []
    Y = []
    for i in range(len(data) - look_back - 1):
        X.append(data[i:(i + look_back), 0])
        Y.append(data[i + look_back, 0])
    return np.array(X), np.array(Y)

X, y = create_sequence(tsla_data_scaled, look_back)

In [ ]:
#Displaying shape of X and y

X.shape, y.shape

((3385, 60), (3385,))

***4. Splitting into Train and Test Data***

In [ ]:
# train-test-split
train_size = int(len(X) * 0.8)
X_train, X_test = X[0:train_size], X[train_size:len(X)]
y_train, y_test = y[0:train_size], y[train_size:len(y)]

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
X_train.shape, y_train.shape, X_test.shape, y_test.shape


((2708, 60, 1), (2708,), (677, 60, 1), (677,))

***5. Building LSTM Model***


In [ ]:
model = keras.Sequential([
     #Adding first LSTM layer with Dropout
     keras.layers.LSTM(units = 200, return_sequences=True, input_shape=(look_back, 1)),
     keras.layers.Dropout(0.2),

     #Adding Second LSTM layer with Dropout
     keras.layers.LSTM(units = 200, return_sequences=True),
     keras.layers.Dropout(0.2),

     #Adding Third LSTM layer with Dropout
     keras.layers.LSTM(units = 200, return_sequences=True),
     keras.layers.Dropout(0.2),

     #Adding Fourth LSTM layer with Dropout
     keras.layers.LSTM(units = 200, return_sequences=True),
     keras.layers.Dropout(0.2),

     #Adding Fifth LSTM layer with Dropout
     keras.layers.LSTM(units = 200, return_sequences=False),
     keras.layers.Dropout(0.2),

     #Adding Dense output layer
     keras.layers.Dense(units = 1)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
#Compilation
model.compile(optimizer = 'adam',
              loss = 'mse',
              metrics = ['RootMeanSquaredError'])

In [ ]:
#Early Stopping Condition
early_stopping = EarlyStopping(monitor = 'val_loss',
                               patience =7,
                               restore_best_weights= True)

In [ ]:
lstm_model = model.fit(X_train, y_train,
                       validation_data=(X_test, y_test),
                       epochs = 200,
                       batch_size = 32,
                       callbacks = [early_stopping],
                       verbose = 1)

Epoch 1/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 20s 76ms/step - RootMeanSquaredError: 0.0754 - loss: 0.0064 - val_RootMeanSquaredError: 0.0873 - val_loss: 0.0076
Epoch 2/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - RootMeanSquaredError: 0.0323 - loss: 0.0011 - val_RootMeanSquaredError: 0.1235 - val_loss: 0.0153
Epoch 3/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - RootMeanSquaredError: 0.0343 - loss: 0.0012 - val_RootMeanSquaredError: 0.0663 - val_loss: 0.0044
Epoch 4/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - RootMeanSquaredError: 0.0329 - loss: 0.0011 - val_RootMeanSquaredError: 0.0747 - val_loss: 0.0056
Epoch 5/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - RootMeanSquaredError: 0.0321 - loss: 0.0010 - val_RootMeanSquaredError: 0.0574 - val_loss: 0.0033
Epoch 6/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - RootMeanSquaredError: 0.0254 - loss: 6.5170e-04 - val_RootMeanSquaredError: 0.1146 - val_loss: 0.0131
Epoch 7/200
85/85 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - RootMeanSquaredError: 0.0416

***6. Making Predictions***

In [ ]:
# Predict stock prices on the test data
tsla_predictions = model.predict(X_test)

#Inverse transform the predictions back to the original price scale
tsla_predictions = tsla_scaler.inverse_transform(tsla_predictions.reshape(-1, 1))


# Inverse transform the actual test data ie y_test back to original price scale
tsla_y_test_rescaled = tsla_scaler.inverse_transform(y_test.reshape(-1, 1))

22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step


***7. Visualizing the results using plotly***

In [ ]:
#Visualizing interactive graphs using plotly
print("\n---  Visualizing TSLA Predictions ---")
#Create a plotly figure

tsla_dates = tsla_data.index[-len(tsla_y_test_rescaled):]
fig = go.Figure()

#Add trace for actual prices
fig.add_trace(go.Scatter(x=tsla_dates, y=tsla_y_test_rescaled.flatten(), mode='lines', name='Actual Price', line=dict(color='blue')))
#Add trace for predicted prices
fig.add_trace(go.Scatter(x=tsla_dates, y=tsla_predictions.flatten(), mode='lines', name='Predicted Price', line=dict(color='red')))

#Add titles and labels
fig.update_layout(title='Tesla Stock Price Prediction', xaxis_title='Date', yaxis_title='Stock Price (USD)', template='plotly_dark')

#Show the figure
fig.show()


---  Visualizing TSLA Predictions ---


***8. Model Evaluation***

In [ ]:
#Calculate MSE, RSME and R2 score
mse = mean_squared_error(tsla_y_test_rescaled, tsla_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(tsla_y_test_rescaled, tsla_predictions)

print(f'Mean Squared Error: {mse}')
print(f'Root Mean Squared Error: {rmse}')
print(f'R2 Score: {r2}')

Mean Squared Error: 131.60520479235703
Root Mean Squared Error: 11.471931171008524
R2 Score: 0.9618514881649392


# ***Processing Google(GOOGL)***

***1. Fetching Data for GOOGL***

In [ ]:
print("---1. Fetching Data for GOOGL---")
# Download GOOGL stock data
googl_stock_data = yf.download('GOOGL', start='2010-01-01', end='2025-01-01')

# Select only the 'Close' column and save it to a CSV file with the date index
googl_data_series = googl_stock_data['Close']
googl_data_series.to_csv("googl_data.csv", header=['Close'])

# Load the data from the simplified CSV file, specifying header and index column
googl_data = pd.read_csv("googl_data.csv", header=0, index_col=0)

# Rename the index to 'Date' for clarity
googl_data.index.name = 'Date'

googl_data.head()

---1. Fetching Data for GOOGL---


/tmp/ipython-input-2247249373.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed


,Close
Date,
2010-01-04,15.576999
2010-01-05,15.508402
2010-01-06,15.117455
2010-01-07,14.765527
2010-01-08,14.962368


***2. Data Preprocessing- EDA***


In [ ]:
print("\n--- 2. Data Preprocessing & EDA for GOOGL ---")
print("\nGOOGL Data Summary:")
print(googl_data.describe())
print("\nGOOGL Data Info:")
print(googl_data.info())

googl_data.isna().sum()


--- 2. Data Preprocessing & EDA for GOOGL ---

GOOGL Data Summary:
             Close
count  3774.000000
mean     61.021261
std      46.540338
min      10.837914
25%      22.534020
50%      47.052727
75%      93.276667
max     196.020935

GOOGL Data Info:
<class 'pandas.core.frame.DataFrame'>
Index: 3774 entries, 2010-01-04 to 2024-12-31
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   3774 non-null   float64
dtypes: float64(1)
memory usage: 59.0+ KB
None


,0
Close,0


In [ ]:
#Remove Duplicates
googl_data=googl_data.drop_duplicates()
googl_data.duplicated().sum()
googl_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3725 entries, 2010-01-04 to 2024-12-31
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   3725 non-null   float64
dtypes: float64(1)
memory usage: 58.2+ KB


***3.Feature Scaling - Data Normalization***


In [ ]:
print("\n--- 3. Feature Scaling -Data Normalization for GOOGL ---")
googl_data_scaled = googl_data.values
googl_scaler = MinMaxScaler(feature_range=(0, 1))
googl_data_scaled = googl_scaler.fit_transform(googl_data_scaled)

googl_data_scaled


--- 3. Feature Scaling -Data Normalization for GOOGL ---


array([[0.02559136],
       [0.02522093],
       [0.02310979],
       ...,
       [0.9790081 ],
       [0.97082676],
       [0.96038473]])

In [ ]:
#Display data in proper Dataframe
Newgoogl_data_scaled = pd.DataFrame(googl_data_scaled, columns = googl_data.columns , index = googl_data.index)
Newgoogl_data_scaled.head()

,Close
Date,
2010-01-04,0.025591
2010-01-05,0.025221
2010-01-06,0.023110
2010-01-07,0.021209
2010-01-08,0.022272


***4.Creating Window Sequences***

In [ ]:
look_back = 60

def create_sequence(data, look_back):
    X = []
    Y = []
    for i in range(len(data) - look_back - 1):
        X.append(data[i:(i + look_back), 0])
        Y.append(data[i + look_back, 0])
    return np.array(X), np.array(Y)

X, y = create_sequence(googl_data_scaled, look_back)

In [ ]:
#Displaying shape of X and y

X.shape, y.shape

((3664, 60), (3664,))

***5. Splitting into Train and Test daaset***

In [ ]:
# train-test-split
train_size = int(len(X) * 0.8)
X_train, X_test = X[0:train_size], X[train_size:len(X)]
y_train, y_test = y[0:train_size], y[train_size:len(y)]

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((2931, 60, 1), (2931,), (733, 60, 1), (733,))

***Building LSTM Model***

In [ ]:
model = keras.Sequential([
     #Adding first LSTM layer with Dropout
     keras.layers.LSTM(units = 200, return_sequences=True, input_shape=(look_back, 1)),
     keras.layers.Dropout(0.2),

     #Adding Second LSTM layer with Dropout
     keras.layers.LSTM(units = 200, return_sequences=True),
     keras.layers.Dropout(0.2),

     #Adding Third LSTM layer with Dropout
     keras.layers.LSTM(units = 200, return_sequences=True),
     keras.layers.Dropout(0.2),

     #Adding Fourth LSTM layer with Dropout
     keras.layers.LSTM(units = 200, return_sequences=True),
     keras.layers.Dropout(0.2),

     #Adding Fifth LSTM layer with Dropout
     keras.layers.LSTM(units = 200, return_sequences=False),
     keras.layers.Dropout(0.2),

     #Adding Dense output layer
     keras.layers.Dense(units = 1)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



In [ ]:
#Compilation
model.compile(optimizer = 'adam',
              loss = 'mse',
              metrics = ['RootMeanSquaredError'])

In [ ]:
#Early Stopping Condition
early_stopping = EarlyStopping(monitor = 'val_loss',
                               patience =7,
                               restore_best_weights= True)

In [ ]:
lstm_model = model.fit(X_train, y_train,
                       validation_data=(X_test, y_test),
                       epochs = 200,
                       batch_size = 32,
                       callbacks = [early_stopping],
                       verbose = 1)

Epoch 1/200
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - RootMeanSquaredError: 0.0814 - loss: 0.0076 - val_RootMeanSquaredError: 0.0447 - val_loss: 0.0020
Epoch 2/200
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - RootMeanSquaredError: 0.0234 - loss: 5.5003e-04 - val_RootMeanSquaredError: 0.0448 - val_loss: 0.0020
Epoch 3/200
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - RootMeanSquaredError: 0.0313 - loss: 9.8755e-04 - val_RootMeanSquaredError: 0.0419 - val_loss: 0.0018
Epoch 4/200
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - RootMeanSquaredError: 0.0256 - loss: 6.6369e-04 - val_RootMeanSquaredError: 0.0661 - val_loss: 0.0044
Epoch 5/200
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - RootMeanSquaredError: 0.0227 - loss: 5.2121e-04 - val_RootMeanSquaredError: 0.0502 - val_loss: 0.0025
Epoch 6/200
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - RootMeanSquaredError: 0.0214 - loss: 4.5788e-04 - val_RootMeanSquaredError: 0.0397 - val_loss: 0.0016
Epoch 7/200
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - RootMeanSquare

***7. Making Predictions***

In [ ]:
# Predict stock prices on the test data
googl_predictions = model.predict(X_test)

#Inverse transform the predictions back to the original price scale
googl_predictions = googl_scaler.inverse_transform(googl_predictions)

# Inverse transform the actual test data ie y_test back to original price scale
googl_y_test_rescaled = googl_scaler.inverse_transform(y_test.reshape(-1, 1))

23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step


***8.Visualizing the results using plotly***

In [ ]:
#Visualizing using plotly
print("\n---  Visualizing GOOGL Predictions ---")
#Create a plotly figure

googl_dates = googl_data.index[-len(googl_y_test_rescaled):]
fig = go.Figure()

#Add trace for actual prices
fig.add_trace(go.Scatter(x=googl_dates, y=googl_y_test_rescaled.flatten(), mode='lines', name='Actual Price', line=dict(color='white')))
#Add trace for predicted prices
fig.add_trace(go.Scatter(x=googl_dates, y=googl_predictions.flatten(), mode='lines', name='Predicted Price', line=dict(color='red')))

#Add titles and labels
fig.update_layout(title='Google Stock Price Prediction', xaxis_title='Date', yaxis_title='Stock Price (USD)', template='plotly_dark')

#Show the figure
fig.show()


---  Visualizing GOOGL Predictions ---


***9.Model Evaluation***

In [ ]:
#Calculate MSE, RSME and R2 score
mse = mean_squared_error(googl_y_test_rescaled, googl_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(googl_y_test_rescaled, googl_predictions)

print(f'Mean Squared Error: {mse}')
print(f'Root Mean Squared Error: {rmse}')
print(f'R2 Score: {r2}')

Mean Squared Error: 30.384981243635295
Root Mean Squared Error: 5.512257363697317
R2 Score: 0.9593059438468271


In [ ]:
#Save the trained models and scalers to files
model.save('tsla_lstm_model.h5')
model.save('googl_lstm_model.h5')

import joblib
joblib.dump(tsla_scaler, 'tsla_scaler.joblib')
joblib.dump(googl_scaler, 'googl_scaler.joblib')

print('Models and scalers saved successfully')

Models and scalers saved successfully


In [34]:
# To mount Google drive
from google.colab import drive

drive.mount('/content/drive')

#save the trained model and scaler to drive
model.save('/content/drive/My Drive/Colab Notebooks/stock_prediction_model/tsla_lstm_model.h5')
joblib.dump(tsla_scaler, '/content/drive/My Drive/Colab Notebooks/stock_prediction_model/tsla_scaler.joblib')

model.save('/content/drive/My Drive/Colab Notebooks/stock_prediction_model/googl_lstm_model.h5')
joblib.dump(tsla_scaler, '/content/drive/My Drive/Colab Notebooks/stock_prediction_model/googl_scaler.joblib')


Mounted at /content/drive


['/content/drive/My Drive/Colab Notebooks/stock_prediction_model/googl_scaler.joblib']

# ***TO CREATE PYTHON SCRIPT FOR STREAMLIT  APP FOR INTERACTIVE DASHBOARD***


In [35]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import load_model
import plotly.graph_objs as go

# ----------------------------
# Load resources once
# ----------------------------
@st.cache_resource
def load_resources():
    tsla_model = load_model("tsla_lstm_model.h5", compile=False)
    tsla_scaler = joblib.load("tsla_scaler.joblib")
    googl_model = load_model("googl_lstm_model.h5", compile=False)
    googl_scaler = joblib.load("googl_scaler.joblib")
    tsla_data = pd.read_csv("tsla_data.csv", parse_dates=True, index_col=0)
    googl_data = pd.read_csv("googl_data.csv", parse_dates=True, index_col=0)

    return {
        "TSLA": {"model": tsla_model, "scaler": tsla_scaler, "data": tsla_data},
        "GOOGL": {"model": googl_model, "scaler": googl_scaler, "data": googl_data},
    }

resources = load_resources()

# ----------------------------
# Streamlit setup
# ----------------------------
st.set_page_config(page_title="📊 Stock Price Prediction", layout="wide")

st.title("📊 Stock Price Prediction Dashboard")
st.markdown("Select a stock, date, and prediction horizon. The app will forecast the closing price for the next few days.")

# ----------------------------
# Sidebar inputs
# ----------------------------
stock_choice = st.sidebar.selectbox("📈 Select Stock", ["TSLA", "GOOGL"])
data = resources[stock_choice]["data"]

# Make sure index is DatetimeIndex and sorted
data = data.sort_index()
data.index = pd.to_datetime(data.index)

# Only allow dates with at least 60 prior trading days
valid_dates = data.index[60:]

selected_date = st.sidebar.date_input(
    "📅 Select Date",
    value=valid_dates[-1],
    min_value=valid_dates[0],
    max_value=valid_dates[-1]
)
selected_date = pd.to_datetime(selected_date)

# ----------------------------
# Helper functions
# ----------------------------
def trading_on_or_before(date: pd.Timestamp) -> pd.Timestamp:
    pos = data.index.searchsorted(date, side="right") - 1
    pos = max(0, min(pos, len(data.index) - 1))
    return data.index[pos]

def trading_n_after(date: pd.Timestamp, n: int) -> pd.Timestamp:
    start_idx = data.index.get_loc(date)
    target_idx = min(start_idx + n, len(data.index) - 1)
    return data.index[target_idx]

# Resolve actual trading day
actual_date = trading_on_or_before(selected_date)

# Horizon setup
idx_actual = data.index.get_loc(actual_date)
days_remaining = (len(data.index) - 1) - idx_actual
max_horizon = min(10, days_remaining)

# ----------------------------
# Slider OR no future days
# ----------------------------
if max_horizon == 0:
    days_ahead = 0
    st.sidebar.info("ℹ️ No predictions available beyond this date.")
else:
    days_ahead = st.sidebar.slider(
        "🔮 Predict how many days ahead?",
        min_value=0,
        max_value=max_horizon,
        value=1
    )

# ----------------------------
# Prediction date logic
# ----------------------------
if days_ahead == 0 and selected_date not in data.index:
    # Special case: weekend/holiday with horizon=0 → shift prediction forward
    prediction_date = trading_n_after(actual_date, 1)
    st.warning(
        f"📌 {selected_date.date()} was **not a trading day**. "
        f"Showing last trading day **{actual_date.date()}** for Actual Close. "
        f"Prediction moved to next trading day **{prediction_date.date()}**."
    )
else:
    prediction_date = trading_n_after(actual_date, days_ahead)

# Alert if selected was adjusted
if selected_date != actual_date and not (days_ahead == 0 and selected_date not in data.index):
    st.warning(
        f"📌 {selected_date.date()} was **not a trading day**. "
        f"Showing last trading day **{actual_date.date()}** for Actual Close."
    )

# Alert if prediction had to skip a holiday/weekend
naive_calendar_pred = selected_date + pd.Timedelta(days=days_ahead)
if prediction_date.date() != naive_calendar_pred.date() and days_ahead > 0:
    st.warning(
        f"📌 The prediction date fell on a **non-trading day**. "
        f"Showing next available trading day **{prediction_date.date()}** instead."
    )

# ----------------------------
# Run prediction
# ----------------------------
scaler = resources[stock_choice]["scaler"]
model = resources[stock_choice]["model"]

past_60 = data["Close"].iloc[idx_actual-60:idx_actual].values.reshape(-1, 1)
scaled_input = scaler.transform(past_60)
last_sequence = scaled_input.copy()

predicted_price = None
if days_ahead > 0 or (days_ahead == 0 and selected_date not in data.index):
    steps = days_ahead if days_ahead > 0 else 1
    for _ in range(steps):
        pred_scaled = model.predict(np.array([last_sequence]), verbose=0)
        last_sequence = np.vstack([last_sequence[1:], pred_scaled])
    predicted_price = float(scaler.inverse_transform(pred_scaled)[0][0])
else:
    predicted_price = float(data.loc[actual_date, "Close"])

actual_close = float(data.loc[actual_date, "Close"])

chg_pct = ((predicted_price - actual_close) / actual_close) * 100 if actual_close else 0.0

# ----------------------------
# Result card
# ----------------------------
# Color logic
if predicted_price > actual_close:
    price_color = "green"
    chg_color = "green"
elif predicted_price < actual_close:
    price_color = "red"
    chg_color = "red"
else:
    price_color = "black"
    chg_color = "black"

st.markdown(
    f"""
    <div style="background:#e3f2fd;padding:20px;border-radius:10px;margin-top:10px;text-align:center;">
        <h2>{stock_choice} Forecast</h2>
        <p><b>Selected Date:</b> {selected_date.date()}</p>
        <p><b>Actual Close ({actual_date.date()}):</b> ${actual_close:.2f}</p>
        <p><b>Predicted Close ({prediction_date.date()}):</b>
            <span style="color:{price_color}; font-weight:bold;">${predicted_price:.2f}</span>
        </p>
        <p><b>Change vs. Last Close:</b>
            <span style="color:{chg_color}; font-weight:bold;">{chg_pct:.2f}%</span>
        </p>
    </div>
    """,
    unsafe_allow_html=True
)



# ----------------------------
# Plot
# ----------------------------
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data.index[:idx_actual+1],
    y=data["Close"].iloc[:idx_actual+1],
    mode="lines",
    name="Actual Price",
    line=dict(color="blue")
))
fig.add_trace(go.Scatter(
    x=[prediction_date],
    y=[predicted_price],
    mode="markers+text",
    name="Predicted Price",
    marker=dict(size=10),
    text=[f"{predicted_price:.2f}"],
    textposition="top center"
))
fig.update_layout(
    title=f"📉 {stock_choice} Historical vs Forecast",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    template="plotly_white",
    plot_bgcolor="#f9fbfd"
)
st.plotly_chart(fig, use_container_width=True)

st.caption("⚠️ Note: Predictions use the last 60 trading days. Longer horizons may be less accurate.")

# ----------------------------
# About Section
# ----------------------------
st.markdown("---")
st.markdown("""
## 📘 About this Dashboard
This Stock-Predictor-Interactive-Dashboard predicts **next-day stock closing prices** for **Tesla (TSLA)** and **Google (GOOGL)**.
It uses a **Long Short-Term Memory (LSTM)** deep learning model, trained on historical stock price data.

### 🔍 How it Works
1. The model takes the **last 60 trading days of closing prices** as input.
2. It learns patterns and trends in stock movements.
3. It outputs a **forecast for the next trading day’s closing price** (or `n` trading days ahead).

### ⚠️ Important Notes
- Markets are volatile; predictions are for **educational purposes only** and **not financial advice**.
- Holidays and weekends are handled automatically using the trading days present in the CSVs.
""")

Writing app.py


In [36]:
# Creating requirement.txt file which contains all the installed packages

%%writefile requirements.txt
streamlit
pyngrok
keras
tensorflow-cpu
datetime
joblib
numpy
pandas
scikit-learn
plotly


Writing requirements.txt
